# 实战案例 2：信息流广告位推荐（Multi-Armed Bandit）

**真实场景**：App 首页 5 个运营位，每个位置可推不同品类活动；用户点击即奖励 1，未点击 0。需在**探索新活动**与**推历史高点击活动**之间平衡。

**本 Notebook**：
1. 模拟 5 臂 Bernoulli Bandit（真实 CTR 未知）
2. 对比 **ε-greedy** 与 **UCB1**
3. 计算累积 Regret

无需 Gymnasium，纯 NumPy 即可运行。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
K = 5  # 5 个广告位/活动
T = 8000  # 8k 次曝光
# 真实 CTR（线上不可见，最优臂 index=2）
TRUE_CTR = np.array([0.04, 0.06, 0.11, 0.05, 0.03])
OPT = TRUE_CTR.max()
print("真实 CTR:", TRUE_CTR, "最优臂:", TRUE_CTR.argmax())

## 1. ε-greedy 策略

In [ ]:
def run_epsilon_greedy(eps=0.1, T=T):
    Q = np.zeros(K); N = np.zeros(K)
    rewards = np.zeros(T); choices = np.zeros(T, dtype=int)
    for t in range(T):
        a = np.random.randint(K) if np.random.rand() < eps else Q.argmax()
        r = 1.0 if np.random.rand() < TRUE_CTR[a] else 0.0
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards[t] = r; choices[t] = a
    regret = np.cumsum(OPT - TRUE_CTR[choices])
    return Q, regret, rewards

Q_eps, regret_eps, rew_eps = run_epsilon_greedy(0.1)
print("ε-greedy 估计 CTR:", np.round(Q_eps, 4))
print("最终累积 Regret:", int(regret_eps[-1]))

## 2. UCB1 策略

In [ ]:
def run_ucb(c=1.0, T=T):
    Q = np.zeros(K); N = np.zeros(K)
    rewards = np.zeros(T); choices = np.zeros(T, dtype=int)
    for t in range(T):
        if t < K:
            a = t  # 每臂至少试一次
        else:
            ucb = Q + c * np.sqrt(np.log(t + 1) / (N + 1e-8))
            a = ucb.argmax()
        r = 1.0 if np.random.rand() < TRUE_CTR[a] else 0.0
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards[t] = r; choices[t] = a
    regret = np.cumsum(OPT - TRUE_CTR[choices])
    return Q, regret, rewards

Q_ucb, regret_ucb, rew_ucb = run_ucb()
print("UCB 估计 CTR:", np.round(Q_ucb, 4))
print("最终累积 Regret:", int(regret_ucb[-1]))

## 3. 纯贪心（反面教材）

In [ ]:
_, regret_greedy, _ = run_epsilon_greedy(eps=0.0)
print("纯贪心 Regret:", int(regret_greedy[-1]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(regret_eps, label="ε-greedy")
axes[0].plot(regret_ucb, label="UCB1")
axes[0].plot(regret_greedy, label="纯贪心", ls="--")
axes[0].set_xlabel("t"); axes[0].set_ylabel("Cumulative Regret"); axes[0].legend(); axes[0].set_title("Regret 对比")
w = 200
axes[1].plot(np.convolve(rew_eps, np.ones(w)/w, "valid"), label="ε-greedy CTR")
axes[1].plot(np.convolve(rew_ucb, np.ones(w)/w, "valid"), label="UCB CTR")
axes[1].axhline(OPT, color="r", ls="--", label="最优臂 CTR")
axes[1].legend(); axes[1].set_title(f"滑动平均 CTR (window={w})")
plt.tight_layout(); plt.show()

## 4. 落地思考

- **难点 1：冷启动** → UCB/Thompson 自动探索；局限：非平稳 CTR 会变
- **难点 2：只优化点击** → 需 Contextual Bandit 加入用户特征；局限：工程复杂度上升
- **难点 3：长期价值** → 需序列 RL + 离线评估；局限：日志偏差

**A/B 上线**：先 5% 流量 shadow，对比 baseline（固定轮播）的 GMV 与 CTR。